
# Pipeline de Pré-processamento de Relatórios

Este notebook orquestra o fluxo de preparação dos relatórios de auditoria:

1. Conversão dos PDFs em imagens
2. Organização dos relatórios por empresa
3. Análise rápida das contagens por classe
4. Anotação manual guiada
5. Geração do dataset anotado em formato JSONL

Os módulos utilizados foram extraídos de `scripts/data_reader` para reaproveitamento em notebooks e scripts.


In [ ]:

from pathlib import Path

from scripts.data_reader.paths import (
    RAW_REPORTS_DIR,
    REPORT_IMAGES_DIR,
    ORGANIZED_REPORTS_DIR,
    ensure_default_directories,
)
from scripts.data_reader.pdf_processing import convert_pdfs_to_images
from scripts.data_reader.report_organizer import organize_reports
from scripts.data_reader.report_stats import (
    count_reports_by_company,
    find_companies_below_threshold,
)
from scripts.data_reader.annotation_tool import annotate_companies
from scripts.data_reader.ocr_dataset import build_dataset_from_reports

print(f"RAW_REPORTS_DIR = {RAW_REPORTS_DIR}")
print(f"REPORT_IMAGES_DIR = {REPORT_IMAGES_DIR}")
print(f"ORGANIZED_REPORTS_DIR = {ORGANIZED_REPORTS_DIR}")


In [ ]:

# Prepara a estrutura de diretórios (executar uma vez)
ensure_default_directories()



## 1. Conversão dos PDFs em imagens

Ative o sinalizador `RUN_CONVERSION` apenas quando novos PDFs forem adicionados em `RAW_REPORTS_DIR`.


In [ ]:

RUN_CONVERSION = False  # altere para True quando quiser reconverter
CONVERSION_DPI = 120

if RUN_CONVERSION:
    convert_pdfs_to_images(RAW_REPORTS_DIR, REPORT_IMAGES_DIR, dpi=CONVERSION_DPI)
    print("Conversão finalizada.")
else:
    print("Conversão ignorada. Ajuste RUN_CONVERSION para True para executar.")



## 2. Organização dos relatórios por empresa

Esta etapa usa OCR para identificar a empresa responsável e mover cada relatório para a estrutura organizada.


In [ ]:

RUN_ORGANIZATION = False

if RUN_ORGANIZATION:
    organize_reports(REPORT_IMAGES_DIR, ORGANIZED_REPORTS_DIR)
    print("Organização concluída.")
else:
    print("Organização ignorada. Ajuste RUN_ORGANIZATION para True para executar.")



## 3. Estatísticas de cobertura

Verifique a quantidade de relatórios por empresa e identifique classes com pouca amostragem.


In [ ]:

MIN_REPORTS_PER_COMPANY = 5

counts = count_reports_by_company(ORGANIZED_REPORTS_DIR)
deficit = find_companies_below_threshold(counts, minimum=MIN_REPORTS_PER_COMPANY)

print("Contagem de relatórios por empresa:
")
for company, total in counts.items():
    print(f"- {company}: {total}")

if deficit:
    print("
Empresas com poucos relatórios:")
    for company, total in deficit.items():
        print(f"- {company}: {total} (mínimo desejado: {MIN_REPORTS_PER_COMPANY})")
else:
    print("
Todas as empresas atingiram o mínimo configurado.")



## 4. Anotação manual guiada

A função abaixo abre uma interface interativa com OpenCV para desenhar caixas. Ajuste `VARIABLES` e `LIMITS` conforme necessário. Execute em lotes para evitar sessões muito longas.


In [ ]:

VARIABLES = [
    "endereço",
    "cod_certificado",
    "validade_certificado",
    "nome_organização",
    "tipo_avaliacao",
    "area_total",
    "producao",
    "data_prox_auditoria",
]

LIMITS = {
    "Control Union": 5,
    "SGS Argentina": 2,
    "Food Chain ID": 5,
    "Genesis Certificações": 3,
    "Bureau Veritas": 3,
    "Cert ID": 2,
}


In [ ]:

RUN_ANNOTATION = False

if RUN_ANNOTATION:
    annotate_companies(
        ORGANIZED_REPORTS_DIR,
        variables=VARIABLES,
        limits=LIMITS,
    )
    print("Anotação encerrada.")
else:
    print("Anotação manual ignorada. Ajuste RUN_ANNOTATION para True para executar.")



## 5. Geração do dataset anotado

Lê os JSONs criados na etapa anterior, roda OCR nas imagens e gera arquivos `.jsonl` rotulados.


In [ ]:

RUN_DATASET_BUILD = False
IOU_THRESHOLD = 0.5
USE_CENTER = False  # alternativo ao IOU

if RUN_DATASET_BUILD:
    build_dataset_from_reports(
        ORGANIZED_REPORTS_DIR,
        iou_threshold=IOU_THRESHOLD,
        use_center=USE_CENTER,
    )
    print("Dataset gerado.")
else:
    print("Construção do dataset ignorada. Ajuste RUN_DATASET_BUILD para True para executar.")



## Próximos passos

- Versão automatizada: combine estas funções em um script (`CLI`) ou workflow (`Makefile`, `Prefect`, `Airflow`).
- Controle de qualidade: criar checagens para validar JSONs gerados (campos obrigatórios, coordenadas válidas, etc.).
- Integração futura: exportar o JSONL final para ferramentas de rotulagem externas (por exemplo, Roboflow, Label Studio) caso seja necessário revisão colaborativa.
